## Set up

### libraries

In [46]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict
from openpyxl.utils import get_column_letter


### Data Extraction

In [47]:
def load_lift_drag_data(root_dir):
    data_by_config_aoa = defaultdict(lambda: {'lift': [], 'drag': [], 'turbulence_model': '', 'geometry': '', 'mesh': '', 'version': ''})
    
    for dirpath, _, filenames in os.walk(root_dir):
        # Find lift and drag files
        lift_files = [f for f in filenames if 'lift_force' in f and f.endswith('.txt')]
        drag_files = [f for f in filenames if 'drag_force' in f and f.endswith('.txt')]
        
        if lift_files and drag_files:
            config = None
            
            # Extract configuration based on method
            if CONFIG_EXTRACTION_METHOD == 'case_file':
                # Look for .cas or .cas.h5 files and extract filename
                for filename in filenames:
                    if filename.endswith('.cas') or filename.endswith('.cas.h5'):
                        config = filename.replace('.cas.h5', '').replace('.cas', '')
                        if config and config[0].isdigit() and '.' in config:
                            break
                        else:
                            config = None
            else:
                # Legacy: parse from folder structure
                parts = dirpath.split(os.sep)
                for part in parts:
                    if part and part[0].isdigit() and part.count('.') >= 2:
                        config = part
                        break
            
            # Find AoA value from folder name (e.g., AoA_10)
            aoa = None
            parts = dirpath.split(os.sep)
            for part in parts:
                if part.startswith('AoA_'):
                    aoa = part
                    break
            
            if config and aoa:
                # Extract AoA number
                aoa_number = aoa.split('_')[1]
                
                # Check if config contains _AoA_ pattern (old format embedded in filename)
                # Example: "4.3.1.3_AoA_5" becomes "4.3.1.3.5"
                if '_AoA_' in config:
                    config = config.replace(f'_AoA_{aoa_number}', f'.{aoa_number}')
                # Check if this is old format with no dots (version name only)
                # Example: "version_name" becomes "version_name.5"
                elif config.count('.') == 0:
                    config = f"{config}.{aoa_number}"
                
                # Parse configuration using position and value mappings
                config_parts = config.split('.')
                positions = POSITION_MAP
                mappings = VALUE_MAPPINGS
                
                # Extract each field based on position
                geometry_num = config_parts[positions['geometry']] if positions['geometry'] is not None and len(config_parts) > positions['geometry'] else None
                mesh_num = config_parts[positions['mesh']] if positions['mesh'] is not None and len(config_parts) > positions['mesh'] else None
                turbulence_num = config_parts[positions['turbulence']] if positions['turbulence'] is not None and len(config_parts) > positions['turbulence'] else None
                version_num = config_parts[positions['version']] if positions['version'] is not None and len(config_parts) > positions['version'] else None
                
                # Map to descriptive names
                geometry = mappings.get('geometry', {}).get(geometry_num, f"Geometry_{geometry_num}") if geometry_num else "N/A"
                mesh = mappings.get('mesh', {}).get(mesh_num, f"Mesh_{mesh_num}") if mesh_num else "N/A"
                turbulence_model = mappings.get('turbulence', {}).get(turbulence_num, f"Turbulence_{turbulence_num}") if turbulence_num else "Unknown"
                version = mappings.get('version', {}).get(version_num, f"Version_{version_num}") if version_num else "N/A"
                
                # Extract angle of attack in degrees (e.g., AoA_10 → 10)
                aoa_degrees = float(aoa_number)
                aoa_radians = np.radians(aoa_degrees)
                
                # Read all lift files
                for lift_file in lift_files:
                    lift_path = os.path.join(dirpath, lift_file)
                    with open(lift_path) as f:
                        lift_data = []
                        for line in f:
                            line = line.strip()
                            # Skip header lines (lines with quotes or non-numeric content)
                            if not line or '"' in line or '(' in line:
                                continue
                            # Split by whitespace and take the second column
                            parts_line = line.split()
                            if len(parts_line) >= 2:
                                try:
                                    value = float(parts_line[1])
                                    lift_data.append(value)
                                except ValueError:
                                    continue
                
                # Read all drag files
                for drag_file in drag_files:
                    drag_path = os.path.join(dirpath, drag_file)
                    with open(drag_path) as f:
                        drag_data = []
                        for line in f:
                            line = line.strip()
                            # Skip header lines (lines with quotes or non-numeric content)
                            if not line or '"' in line or '(' in line:
                                continue
                            # Split by whitespace and take the second column
                            parts_line = line.split()
                            if len(parts_line) >= 2:
                                try:
                                    value = float(parts_line[1])
                                    drag_data.append(value)
                                except ValueError:
                                    continue
                
                # Apply angle of attack correction to transform forces
                # True lift = lift*cos(theta) - drag*sin(theta)
                # True drag = lift*sin(theta) + drag*cos(theta)
                cos_theta = np.cos(aoa_radians)
                sin_theta = np.sin(aoa_radians)
                
                true_lift_data = []
                true_drag_data = []
                
                # Ensure both arrays have the same length
                min_length = min(len(lift_data), len(drag_data))
                for i in range(min_length):
                    true_lift = lift_data[i] * cos_theta - drag_data[i] * sin_theta
                    true_drag = lift_data[i] * sin_theta + drag_data[i] * cos_theta
                    true_lift_data.append(true_lift)
                    true_drag_data.append(true_drag)
                
                # Store the corrected data
                data_by_config_aoa[(config, aoa)]['lift'].extend(true_lift_data)
                data_by_config_aoa[(config, aoa)]['drag'].extend(true_drag_data)
                
                # Store metadata
                data_by_config_aoa[(config, aoa)]['geometry'] = geometry
                data_by_config_aoa[(config, aoa)]['mesh'] = mesh
                data_by_config_aoa[(config, aoa)]['turbulence_model'] = turbulence_model
                data_by_config_aoa[(config, aoa)]['version'] = version
    
    return data_by_config_aoa


In [48]:
def compute_statistics(data):
    mean_val = np.mean(data)
    std_dev = np.std(data)
    # Calculate Coefficient of Variation (COV) as percentage
    cov = (std_dev / mean_val * 100) if mean_val != 0 else 0
    return mean_val, cov

## Enter Values Here

In [49]:
# ==================== CONFIGURATION ====================
# User parameters - modify these as needed

base_path = r"C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3"
output_dir = r"C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3"
num_iterations = 150  # Number of latest iterations to use for statistics

# Select configuration extraction method:
# 'case_file'   = Read config directly from .cas/.cas.h5 filename (RECOMMENDED)
# 'folder_path' = Parse config from folder structure (legacy method)
CONFIG_EXTRACTION_METHOD = 'case_file'

# ==================== POSITION MAPPING ====================
# Tell the code which position in the config corresponds to which field
# Example: For config "4.3.1.2.5", specify which number is mesh, turbulence, AoA, etc.
POSITION_MAP = {
    'geometry': 0,      # Position 0 = first number (4 in "4.3.1.2.5")
    'mesh': 1,          # Position 1 = second number (3 in "4.3.1.2.5")
    'turbulence': 2,    # Position 2 = third number (1 in "4.3.1.2.5")
    'version': 3,       # Position 3 = fourth number (2 in "4.3.1.2.5")
    'aoa': 4,           # Position 4 = fifth number (5 in "4.3.1.2.5") - NEW FORMAT ONLY
    'status': None      # Position for status field (or None if not present)
}

# ==================== VALUE MAPPINGS ====================
# Define what each number means for each position
VALUE_MAPPINGS = {
    'geometry': {
        '1': 'Baseline Geometry',
        '2': 'Modified Geometry',
        '3': 'Grid Fins Geometry',
        '4': 'Baseline'
    },
    'mesh': {
        '1': 'Coarse Mesh',
        '2': 'Medium Mesh',
        '3': 'Standard',
        '4': 'NM Adjusted'
    },
    'turbulence': {
        '1': 'SST',
        '2': 'RNG',
        '3': 'RSM',
        '4': 'k-epsilon'
    },
    'version': {
        '1': 'Version 1',
        '2': 'Version 2',
        '3': 'Version 3',
        '4': 'Version 4'
    },
    'aoa': {
        '0': '0 degrees',
        '1': '1 degrees',
        '2': '2 degrees',
        '3': '3 degrees',
        '4': '4 degrees',
        '5': '5 degrees',
        '6': '6 degrees',
        '7': '7 degrees',
        '8': '8 degrees',
        '9': '9 degrees',
        '10': '10 degrees',
        '11': '11 degrees',
        '12': '12 degrees',
        '13': '13 degrees',
        '14': '14 degrees',
        '15': '15 degrees',
        '16': '16 degrees',
        '17': '17 degrees',
        '18': '18 degrees',
        '19': '19 degrees',
        '20': '20 degrees',
        '21': '21 degrees',
        '22': '22 degrees',
        '23': '23 degrees',
        '24': '24 degrees',
        '25': '25 degrees'
    }
}

# Display current configuration
print("=" * 80)
print("CONFIGURATION SETTINGS:")
print("=" * 80)
print(f"✓ Data Path: {base_path}")
print(f"✓ Output Directory: {output_dir}")
print(f"✓ Iterations for Statistics: {num_iterations}")
print(f"✓ Config Extraction: {CONFIG_EXTRACTION_METHOD}")
if CONFIG_EXTRACTION_METHOD == 'case_file':
    print(f"  → Reading configuration from .cas file")
    print(f"  → Position mapping: geometry={POSITION_MAP['geometry']}, mesh={POSITION_MAP['mesh']}, turbulence={POSITION_MAP['turbulence']}, version={POSITION_MAP['version']}, aoa={POSITION_MAP['aoa']}")
else:
    print(f"  → Parsing from folder structure (legacy mode)")
print("=" * 80)

# ==================== PROCESSING ====================

# Load all data
all_data = load_lift_drag_data(base_path)

# Create output directories
os.makedirs(output_dir, exist_ok=True)
raw_data_dir = os.path.join(output_dir, "raw_data")
os.makedirs(raw_data_dir, exist_ok=True)

# Sort AoA values numerically (5, 10, 15, 20)
def extract_aoa_number(aoa_str):
    return int(aoa_str.split('_')[1])

# Export each dataset with a unique name based on config only (config already includes AoA)
for (config, aoa), data in all_data.items():
    # Use config as filename base since it already includes the AoA (e.g., "4.3.1.3.5")
    filename_base = config
    
    # Export lift data to raw_data folder
    lift_output = os.path.join(raw_data_dir, f"{filename_base}_lift.txt")
    with open(lift_output, 'w') as f:
        for value in data['lift']:
            f.write(f"{value}\n")
    
    # Export drag data to raw_data folder
    drag_output = os.path.join(raw_data_dir, f"{filename_base}_drag.txt")
    with open(drag_output, 'w') as f:
        for value in data['drag']:
            f.write(f"{value}\n")
    
    print(f"Exported: {filename_base}")

# Create summary statistics file using specified number of iterations
summary_file = os.path.join(output_dir, "SUMMARY_Statistics.txt")

with open(summary_file, 'w') as f:
    f.write("=" * 100 + "\n")
    f.write(f"SUMMARY STATISTICS - Last {num_iterations} Iterations\n")
    f.write("=" * 100 + "\n\n")
    
    # Sort by config and then by AoA numerically
    sorted_data = sorted(all_data.items(), key=lambda x: (x[0][0], extract_aoa_number(x[0][1])))
    
    for (config, aoa), data in sorted_data:
        # Get last N iterations (or all if less than N)
        lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
        drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
        
        # Calculate statistics
        lift_mean = np.mean(lift_last_n) if lift_last_n else 0
        lift_std = np.std(lift_last_n) if lift_last_n else 0
        lift_cov = (lift_std / lift_mean * 100) if lift_mean != 0 else 0
        
        drag_mean = np.mean(drag_last_n) if drag_last_n else 0
        drag_std = np.std(drag_last_n) if drag_last_n else 0
        drag_cov = (drag_std / drag_mean * 100) if drag_mean != 0 else 0
        
        # Write to file
        f.write(f"Configuration: {config}\n")
        f.write(f"Turbulence Model: {data['turbulence_model']}\n")
        f.write(f"Version: {data['version']}\n")
        f.write(f"Angle of Attack: {aoa}\n")
        f.write(f"Iterations used: {len(lift_last_n)}\n\n")
        f.write(f"LIFT:\n")
        f.write(f"  Average: {lift_mean:12.6f}\n")
        f.write(f"  COV (%): {lift_cov:12.6f}\n\n")
        f.write(f"DRAG:\n")
        f.write(f"  Average: {drag_mean:12.6f}\n")
        f.write(f"  COV (%): {drag_cov:12.6f}\n")
        f.write("-" * 100 + "\n\n")
        f.write(f"LIFT:\n")
        f.write(f"  Average: {lift_mean:12.6f}\n")

CONFIGURATION SETTINGS:
✓ Data Path: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3
✓ Output Directory: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3
✓ Iterations for Statistics: 150
✓ Config Extraction: case_file
  → Reading configuration from .cas file
  → Position mapping: geometry=0, mesh=1, turbulence=2, version=3, aoa=4
Exported: 4.3.1.2.10
Exported: 4.3.1.2.15
Exported: 4.3.1.2.20
Exported: 4.3.1.2.5
Exported: 4.3.1.3.10
Exported: 4.3.1.3.15
Exported: 4.3.1.3.20
Exported: 4.3.1.3.5
Exported: 4.3.1.4.10
Exported: 4.3.1.4.15
Exported: 4.3.1.4.20
Exported: 4.3.1.4.5
Exported: 4.3.2.1.10
Exported: 4.3.2.1.15
Exported: 4.3.2.1.5
Exported: 4.3.3.1.10
Exported: 4.3.3.1.15
Exported: 4.3.3.1.20
Exported: 4.3.3.1.5
Exported: 4.3.3.1.15
Exported: 4.3.3.1.20
Exported: 4.3.3.1.5


In [50]:
# ==================== CREATE BAR GRAPHS ====================

graphs_dir = os.path.join(output_dir, "graphs")
os.makedirs(graphs_dir, exist_ok=True)

# Group data by AoA and turbulence model
aoa_groups = defaultdict(lambda: {'SST': {'lift': [], 'drag': []}, 'RNG': {'lift': [], 'drag': []}, 'RMS': {'lift': [], 'drag': []}})

for (config, aoa), data in all_data.items():
    turbulence_model = data['turbulence_model']
    
    # Get last N iterations for graphing
    lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
    drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
    
    # Calculate means
    lift_mean = np.mean(lift_last_n) if lift_last_n else 0
    drag_mean = np.mean(drag_last_n) if drag_last_n else 0
    
    if turbulence_model in aoa_groups[aoa]:
        aoa_groups[aoa][turbulence_model]['lift'].append(lift_mean)
        aoa_groups[aoa][turbulence_model]['drag'].append(drag_mean)

# Create bar graphs for each AoA
for aoa in sorted(aoa_groups.keys(), key=extract_aoa_number):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Prepare data for plotting
    models = ['SST', 'RNG', 'RMS']
    lift_means = []
    drag_means = []
    
    for model in models:
        if aoa_groups[aoa][model]['lift']:
            lift_means.append(np.mean(aoa_groups[aoa][model]['lift']))
        else:
            lift_means.append(0)
        
        if aoa_groups[aoa][model]['drag']:
            drag_means.append(np.mean(aoa_groups[aoa][model]['drag']))
        else:
            drag_means.append(0)
    
    # Create bar charts
    x = np.arange(len(models))
    width = 0.6
    
    ax1.bar(x, lift_means, width, color=['#1f77b4', '#ff7f0e', '#2ca02c'], edgecolor='black', linewidth=1.5)
    ax1.set_title(f'Lift Comparison - {aoa}', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Lift Mean Value', fontsize=12)
    ax1.set_xticks(x)
    ax1.set_xticklabels(models, fontsize=11)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(lift_means):
        ax1.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    
    ax2.bar(x, drag_means, width, color=['#1f77b4', '#ff7f0e', '#2ca02c'], edgecolor='black', linewidth=1.5)
    ax2.set_title(f'Drag Comparison - {aoa}', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Drag Mean Value', fontsize=12)
    ax2.set_xticks(x)
    ax2.set_xticklabels(models, fontsize=11)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(drag_means):
        ax2.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    
    # Save graph
    aoa_num = extract_aoa_number(aoa)
    graph_file = os.path.join(graphs_dir, f"turbulence_comparison_AoA_{aoa_num}.png")
    plt.savefig(graph_file, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Created bar graph for {aoa}: {graph_file}")

print(f"\n✓ All bar graphs saved to: {graphs_dir}")
print(f"✓ Note: Graphs exclude data with Adapted=Yes or Grid Fins=Yes")


Created bar graph for AoA_5: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3\graphs\turbulence_comparison_AoA_5.png
Created bar graph for AoA_10: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3\graphs\turbulence_comparison_AoA_10.png
Created bar graph for AoA_10: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3\graphs\turbulence_comparison_AoA_10.png
Created bar graph for AoA_15: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3\graphs\turbulence_comparison_AoA_15.png
Created bar graph for AoA_15: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3\graphs\turbulence_comparison_AoA_15.png
Created bar graph for AoA_20: C:\Users\lukek\OneDrive - Liberty University\Gr

In [51]:
# ==================== CREATE EXCEL TABLES ====================

from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# Create Excel file with all data
excel_file = os.path.join(output_dir, "SUMMARY_Statistics.xlsx")

# Recreate config_groups for reference by turbulence model
config_groups = defaultdict(lambda: {'lift': {}, 'drag': {}})
for (config, aoa), data in all_data.items():
    config_parts = config.split('.')
    # Get turbulence number from position specified in POSITION_MAP
    turbulence_pos = POSITION_MAP['turbulence']
    turbulence_num = config_parts[turbulence_pos] if turbulence_pos is not None and len(config_parts) > turbulence_pos else "unknown"
    
    lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
    drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
    
    config_groups[turbulence_num]['lift'][aoa] = lift_last_n
    config_groups[turbulence_num]['drag'][aoa] = drag_last_n

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    # Create a summary sheet with all configurations
    summary_data = []
    
    # Sort by config and then by AoA numerically
    sorted_data = sorted(all_data.items(), key=lambda x: (x[0][0], extract_aoa_number(x[0][1])))
    
    for (config, aoa), data in sorted_data:
        # Get last N iterations
        lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
        drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
        
        # Calculate statistics
        lift_mean = np.mean(lift_last_n) if lift_last_n else 0
        lift_std = np.std(lift_last_n) if lift_last_n else 0
        lift_cov = (lift_std / lift_mean * 100) if lift_mean != 0 else 0
        
        drag_mean = np.mean(drag_last_n) if drag_last_n else 0
        drag_std = np.std(drag_last_n) if drag_last_n else 0
        drag_cov = (drag_std / drag_mean * 100) if drag_mean != 0 else 0
        
        summary_data.append({
            'Configuration': config,
            'Turbulence Model': data['turbulence_model'],
            'Version': data['version'],
            'Angle of Attack': aoa,
            'Iterations Used': len(lift_last_n),
            'Lift Mean (N)': f"{lift_mean:.3f}",
            'Lift COV (%)': f"{lift_cov:.2f}",
            'Drag Mean (N)': f"{drag_mean:.3f}",
            'Drag COV (%)': f"{drag_cov:.2f}"
        })
    
    # Write summary to main sheet
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    
    # Create consolidated Turbulence Models sheet
    turbulence_models_data = []
    
    for turbulence_num in sorted(config_groups.keys()):
        # Sort AoA numerically within each turbulence model type
        sorted_aoas = sorted(config_groups[turbulence_num]['lift'].keys(), key=extract_aoa_number)
        
        for aoa in sorted_aoas:
            lift_data = config_groups[turbulence_num]['lift'][aoa]
            drag_data = config_groups[turbulence_num]['drag'][aoa]
            
            # Get turbulence model info (should be same for all in this group)
            turbulence_model = ""
            version = ""
            for (config, config_aoa), d in all_data.items():
                if config_aoa == aoa:
                    config_parts = config.split('.')
                    turb_pos = POSITION_MAP['turbulence']
                    if turb_pos is not None and len(config_parts) > turb_pos and config_parts[turb_pos] == turbulence_num:
                        turbulence_model = d['turbulence_model']
                        version = d['version']
                        break
            
            # Calculate COV for lift and drag
            lift_mean_val = np.mean(lift_data)
            lift_std_val = np.std(lift_data)
            lift_cov_val = (lift_std_val / lift_mean_val * 100) if lift_mean_val != 0 else 0
            
            drag_mean_val = np.mean(drag_data)
            drag_std_val = np.std(drag_data)
            drag_cov_val = (drag_std_val / drag_mean_val * 100) if drag_mean_val != 0 else 0
            
            turbulence_models_data.append({
                'Turbulence Model': turbulence_model,
                'Version': version,
                'Angle of Attack': aoa,
                'Lift Mean (N)': f"{lift_mean_val:.3f}",
                'Lift COV (%)': f"{lift_cov_val:.2f}",
                'Drag Mean (N)': f"{drag_mean_val:.3f}",
                'Drag COV (%)': f"{drag_cov_val:.2f}",
                'Num Points': len(lift_data)
            })
    
    turbulence_models_df = pd.DataFrame(turbulence_models_data)
    turbulence_models_df.to_excel(writer, sheet_name='Turbulence_Models', index=False)

# Apply formatting to all sheets
from openpyxl import load_workbook
wb = load_workbook(excel_file)

# Define styles
header_font = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
header_fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
header_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

data_alignment = Alignment(horizontal='center', vertical='center')
border_style = Border(
    left=Side(style='thin', color='000000'),
    right=Side(style='thin', color='000000'),
    top=Side(style='thin', color='000000'),
    bottom=Side(style='thin', color='000000')
)

# Alternating row colors
row_fill_light = PatternFill(start_color='F2F2F2', end_color='F2F2F2', fill_type='solid')
row_fill_white = PatternFill(start_color='FFFFFF', end_color='FFFFFF', fill_type='solid')

for ws_name in wb.sheetnames:
    worksheet = wb[ws_name]
    
    # Autofit column widths
    for column in worksheet.columns:
        max_length = 0
        column_letter = get_column_letter(column[0].column)
        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = min(max_length + 2, 50)
        worksheet.column_dimensions[column_letter].width = adjusted_width
    
    # Format header row
    for cell in worksheet[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = border_style
    
    # Format data rows with alternating colors and borders
    for row_idx, row in enumerate(worksheet.iter_rows(min_row=2), start=2):
        fill_color = row_fill_light if row_idx % 2 == 0 else row_fill_white
        for cell in row:
            cell.alignment = data_alignment
            cell.border = border_style
            cell.fill = fill_color
    
    # Set row height for header
    worksheet.row_dimensions[1].height = 30

wb.save(excel_file)

print(f"✓ Excel file created: {excel_file}")
print(f"✓ Sheet names: Summary, Turbulence_Models")
print(f"✓ Professional formatting applied: colored headers, borders, alternating rows")
print(f"✓ All lift and drag values displayed in Newtons (N)")

✓ Excel file created: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3\SUMMARY_Statistics.xlsx
✓ Sheet names: Summary, Turbulence_Models
✓ Professional formatting applied: colored headers, borders, alternating rows
✓ All lift and drag values displayed in Newtons (N)


In [ ]:
# ==================== CREATE TURBULENCE MODEL COMPARISON TABLE ====================

import time

# Create a comparison table for turbulence models
turbulence_comparison = []

# Get all unique AoAs
all_aoas = set()
for (config, aoa), data in all_data.items():
    all_aoas.add(aoa)

# Sort AoAs numerically
sorted_aoas = sorted(all_aoas, key=extract_aoa_number)

# Dynamically determine which turbulence models are available
all_models = set()
for (config, aoa), data in all_data.items():
    all_models.add(data['turbulence_model'])

print(f"✓ Turbulence models in dataset: {sorted(all_models)}")

# For each AoA, get data for each turbulence model
for aoa in sorted_aoas:
    aoa_entry = {'Angle of Attack': aoa}
    
    # Collect lift and drag means and COV for each model
    lift_means = {}
    drag_means = {}
    lift_covs = {}
    drag_covs = {}
    
    for model in sorted(all_models):
        model_lift_values = []
        model_drag_values = []
        
        # Find all configs with this AOA and turbulence model
        for (config, config_aoa), data in all_data.items():
            if config_aoa == aoa and data['turbulence_model'] == model:
                lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
                drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
                
                if lift_last_n:
                    model_lift_values.extend(lift_last_n)
                if drag_last_n:
                    model_drag_values.extend(drag_last_n)
        
        # Calculate means and COV for this model
        if model_lift_values:
            lift_means[model] = np.mean(model_lift_values)
            lift_std = np.std(model_lift_values)
            lift_covs[model] = (lift_std / lift_means[model] * 100) if lift_means[model] != 0 else 0
        else:
            lift_means[model] = None
            lift_covs[model] = None
        
        if model_drag_values:
            drag_means[model] = np.mean(model_drag_values)
            drag_std = np.std(model_drag_values)
            drag_covs[model] = (drag_std / drag_means[model] * 100) if drag_means[model] != 0 else 0
        else:
            drag_means[model] = None
            drag_covs[model] = None
    
    # Add lift values with COV and percent difference from SST
    lift_str = ""
    sst_lift = lift_means.get('SST')
    for model in sorted(all_models):
        if lift_means[model] is not None and lift_covs[model] is not None:
            if lift_str:
                lift_str += " | "
            # Calculate percent difference from SST
            if model != 'SST' and sst_lift is not None and sst_lift != 0:
                pct_diff = ((lift_means[model] - sst_lift) / sst_lift * 100)
                lift_str += f"{model}: {lift_means[model]:.3f} N ({lift_covs[model]:.2f}%, {pct_diff:+.2f}% vs SST)"
            else:
                lift_str += f"{model}: {lift_means[model]:.3f} N ({lift_covs[model]:.2f}%)"
        else:
            if lift_str:
                lift_str += " | "
            lift_str += f"{model}: N/A"
    aoa_entry['Lift (N)'] = lift_str
    
    # Add drag values with COV and percent difference from SST
    drag_str = ""
    sst_drag = drag_means.get('SST')
    for model in sorted(all_models):
        if drag_means[model] is not None and drag_covs[model] is not None:
            if drag_str:
                drag_str += " | "
            # Calculate percent difference from SST
            if model != 'SST' and sst_drag is not None and sst_drag != 0:
                pct_diff = ((drag_means[model] - sst_drag) / sst_drag * 100)
                drag_str += f"{model}: {drag_means[model]:.3f} N ({drag_covs[model]:.2f}%, {pct_diff:+.2f}% vs SST)"
            else:
                drag_str += f"{model}: {drag_means[model]:.3f} N ({drag_covs[model]:.2f}%)"
        else:
            if drag_str:
                drag_str += " | "
            drag_str += f"{model}: N/A"
    aoa_entry['Drag (N)'] = drag_str
    
    turbulence_comparison.append(aoa_entry)

# Add to existing Excel file
comparison_df = pd.DataFrame(turbulence_comparison)

# Write comparison sheet
with pd.ExcelWriter(excel_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    comparison_df.to_excel(writer, sheet_name='Turbulence_Model_Comparison', index=False)

# Apply formatting to Turbulence_Model_Comparison sheet
time.sleep(0.5)  # Brief delay to ensure file is fully released

from openpyxl.styles.colors import Color

wb = load_workbook(excel_file)
ws = wb['Turbulence_Model_Comparison']

# Define styles
header_font = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
header_fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
header_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
data_alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
border_style = Border(
    left=Side(style='thin', color='000000'),
    right=Side(style='thin', color='000000'),
    top=Side(style='thin', color='000000'),
    bottom=Side(style='thin', color='000000')
)
row_fill_light = PatternFill(start_color='F2F2F2', end_color='F2F2F2', fill_type='solid')
row_fill_white = PatternFill(start_color='FFFFFF', end_color='FFFFFF', fill_type='solid')

# Color coding for highlights
cov_color = Color(rgb='FF0000')  # Red for COV
pct_diff_color = Color(rgb='4169E1')  # Royal blue for percentage comparison

# Autofit column widths
for column in ws.columns:
    max_length = 0
    column_letter = get_column_letter(column[0].column)
    for cell in column:
        try:
            if len(str(cell.value)) > max_length:
                max_length = len(str(cell.value))
        except:
            pass
    adjusted_width = min(max_length + 2, 80)
    ws.column_dimensions[column_letter].width = adjusted_width

# Format header row with legend
for cell in ws[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = header_alignment
    cell.border = border_style

# Add legend row below header to explain format
ws.insert_rows(2)
legend_font = Font(name='Calibri', size=9, italic=True)
legend_fill = PatternFill(start_color='E0E0E0', end_color='E0E0E0', fill_type='solid')

ws['A2'] = 'Legend'
ws['B2'] = 'Format: Model: Mean N (COV%, %Diff vs SST) | Red=COV | Blue=%Difference'
ws['A2'].font = legend_font
ws['B2'].font = legend_font
ws['A2'].fill = legend_fill
ws['B2'].fill = legend_fill
ws['A2'].border = border_style
ws['B2'].border = border_style
ws['A2'].alignment = Alignment(horizontal='center', vertical='center')
ws['B2'].alignment = Alignment(horizontal='left', vertical='center')
ws.merge_cells('B2:C2')

# Format data rows with alternating colors, borders, and highlighting
for row_idx, row in enumerate(ws.iter_rows(min_row=3), start=3):
    fill_color = row_fill_light if row_idx % 2 == 1 else row_fill_white
    for cell_idx, cell in enumerate(row):
        cell.alignment = data_alignment
        cell.border = border_style
        cell.fill = fill_color
        
        # Apply color highlighting to Lift and Drag columns (columns B and C)
        if cell_idx in [1, 2] and cell.value:  # Columns B and C (Lift and Drag)
            cell_text = str(cell.value)
            from openpyxl.cell.text import InlineFont
            from openpyxl.cell.rich_text import TextBlock, CellRichText
            
            # Parse and create rich text with colored segments
            # Format is: "Model: Value N (COV%, +X% vs SST)" or "Model: Value N (COV%)"
            text_parts = []
            i = 0
            while i < len(cell_text):
                # Check for opening parenthesis
                if cell_text[i] == '(':
                    # Find the closing parenthesis
                    closing_paren = cell_text.find(')', i)
                    if closing_paren != -1:
                        segment = cell_text[i:closing_paren+1]
                        
                        # Check if this segment contains "vs SST"
                        if 'vs SST' in segment:
                            # This segment has both COV and percentage difference
                            # Split by comma to separate them
                            if ',' in segment:
                                parts = segment[1:-1].split(',', 1)  # Remove outer parentheses and split once
                                # First part is COV (e.g., "3.45%")
                                cov_part = f"({parts[0].strip()})"
                                text_parts.append(TextBlock(InlineFont(color=cov_color), cov_part))
                                # Second part is percentage diff (e.g., " +1.96% vs SST")
                                pct_part = f"({parts[1].strip()})"
                                text_parts.append(TextBlock(InlineFont(color=pct_diff_color), pct_part))
                            else:
                                # Shouldn't happen, but just in case
                                text_parts.append(TextBlock(InlineFont(color=pct_diff_color), segment))
                        elif '%' in segment:
                            # This is just COV percentage - color it red
                            text_parts.append(TextBlock(InlineFont(color=cov_color), segment))
                        else:
                            # No percentage, regular text
                            text_parts.append(TextBlock(InlineFont(), segment))
                        
                        i = closing_paren + 1
                        continue
                
                # Regular text (not in parentheses)
                j = i
                while j < len(cell_text) and cell_text[j] != '(':
                    j += 1
                if j > i:
                    text_parts.append(TextBlock(InlineFont(), cell_text[i:j]))
                i = j
            
            if text_parts:
                cell.value = CellRichText(*text_parts)

# Set row height for header and legend
ws.row_dimensions[1].height = 30
ws.row_dimensions[2].height = 25

wb.save(excel_file)
wb.close()

print(f"\n✓ Turbulence Model Comparison table added to Excel file")
print(f"✓ Format: Model: Mean (COV%) with percent difference from SST")
print(f"✓ Professional formatting applied to comparison sheet")
print(f"\nSample comparison:")
for row in turbulence_comparison:
    print(f"\n{row['Angle of Attack']}:")
    print(f"  Lift (N):  {row['Lift (N)']}")
    print(f"  Drag (N):  {row['Drag (N)']}")


IndentationError: unindent does not match any outer indentation level (<string>, line 41)